# 🚀 PI-DeepONet-RAR: Kaggle Execution Notebook (Dataset / Git Workflow)

Welcome to the official Kaggle execution notebook for **PI-DeepONet-RAR**.

### 📌 How this notebook works with your Kaggle Dataset:
1. **Git / Dataset Integration**: You push your code to Git and add it to Kaggle as a Dataset (located under `/kaggle/input/`).
2. **Read-Only Fix**: Since `/kaggle/input/` is read-only, this code automatically writes all generated model checkpoints (`.pt`), CSV logs, and plots to `/kaggle/working/results`.
3. **GPU Accelerator**: Runs on Kaggle's free GPU (P100 / T4).
4. **Full Analytics**: Renders loss curves, adaptive data growth, validation error tables, and 2D stress field contour maps directly in the notebook!

In [ ]:
# ========================================================
# 1. GPU Verification & Dependencies Setup
# ========================================================
import os
import sys
import torch

print("="*60)
print("  Kaggle Environment & GPU Verification")
print("="*60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device:    {device}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Make sure GPU is enabled in Kaggle Settings -> Accelerator -> GPU P100/T4.")

# Install dependencies if needed
!pip install -q torch numpy matplotlib scipy pyyaml pandas


In [ ]:
# ========================================================
# 2. Locate Kaggle Dataset & Configure Paths
# ========================================================
import glob

# Automatically find project root in /kaggle/input or current workspace
possible_roots = [
    "/kaggle/input/*pi-deeponet*",
    "/kaggle/input/*sciml*",
    "/kaggle/input/*",
    ".",
    ".."
]

project_root = None
for pattern in possible_roots:
    matches = glob.glob(pattern)
    for m in matches:
        if os.path.exists(os.path.join(m, "kaggle_run_all.py")):
            project_root = os.path.abspath(m)
            break
    if project_root:
        break

if project_root:
    print(f"✅ Found Project Root Dataset at: {project_root}")
else:
    project_root = os.getcwd()
    print(f"⚠️ Could not automatically detect dataset directory. Using current dir: {project_root}")

# Ensure modules are discoverable
for path_to_add in [project_root, os.path.join(project_root, 'src'), os.path.join(project_root, 'fem_baseline')]:
    if path_to_add not in sys.path:
        sys.path.insert(0, path_to_add)

working_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
results_dir = os.path.join(working_dir, "results")
os.makedirs(results_dir, exist_ok=True)

print(f"Output Results Directory: {results_dir}")


## 🏋️‍♂️ 3. Execute Training Experiments
Choose execution mode and run all 4 experimental arms (`baseline`, `rar_collocation`, `rar_load`, `rar_combined`).
- **`QUICK_RUN = True`**: Fast test run (~3-5 mins on Kaggle GPU).
- **`QUICK_RUN = False`**: Full scientific benchmark (~1-2 hours on Kaggle GPU).

In [ ]:
# ========================================================
# 3. Run Training Suite
# ========================================================
QUICK_RUN = True   # Set to False for full experiment training
ARMS_TO_RUN = ["baseline", "rar_collocation", "rar_load", "rar_combined"]

import subprocess

runner_script = os.path.join(project_root, "kaggle_run_all.py")
cmd = [sys.executable, runner_script]
if QUICK_RUN:
    cmd.append("--quick")
cmd.extend(["--arms"] + ARMS_TO_RUN)

print("="*60)
print(f"  Starting Experiment Runner on Kaggle GPU...")
print(f"  Script: {runner_script}")
print(f"  Mode:   {'Quick (~5 min test)' if QUICK_RUN else 'Full Experiment'}")
print(f"  Arms:   {', '.join(ARMS_TO_RUN)}")
print("="*60 + "\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=working_dir)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    print("\n🎉 Training and validation completed successfully!")
else:
    print(f"\n⚠️ Execution finished with exit code {process.returncode}")


## 📊 4. Training Loss & RAR Growth Analytics
Displays loss convergence comparison and adaptive points/loads growth generated during training.

In [ ]:
# ========================================================
# 4. Summary Plots
# ========================================================
from IPython.display import Image, display

plots_dir = os.path.join(results_dir, "plots")
loss_plot = os.path.join(plots_dir, "loss_comparison.png")
growth_plot = os.path.join(plots_dir, "rar_growth.png")
scf_plot = os.path.join(plots_dir, "scf_comparison.png")

if os.path.exists(loss_plot):
    print("📈 Training Loss Comparison Across Arms:")
    display(Image(filename=loss_plot))

if os.path.exists(growth_plot):
    print("\n🌱 Adaptive Data Growth (RAR Points & Loads):")
    display(Image(filename=growth_plot))

if os.path.exists(scf_plot):
    print("\n🎯 Stress Concentration Factor (SCF) Accuracy:")
    display(Image(filename=scf_plot))


## 📋 5. Quantitative Validation Summary
Relative $L_2$ error table comparing PI-DeepONet stress predictions against the exact analytical Kirsch solution.

In [ ]:
# ========================================================
# 5. Validation Error Table
# ========================================================
import pandas as pd

val_csv = os.path.join(results_dir, "validation_results.csv")
if os.path.exists(val_csv):
    df_val = pd.read_csv(val_csv, index_col=0)
    print("=== Validation Metrics vs Exact Kirsch Solution ===")
    display(df_val.style.format({
        "l2_sxx": "{:.4e}",
        "l2_syy": "{:.4e}",
        "l2_sxy": "{:.4e}",
        "scf": "{:.4f}"
    }).highlight_min(axis=0, color="lightgreen"))
else:
    print("⚠️ Validation results file not found.")


## 🎨 6. 2D Stress Field Verification Maps
High-resolution spatial contour comparison between model prediction and theoretical Kirsch stress fields.

In [ ]:
# ========================================================
# 6. 2D Stress Field Contour Visualization
# ========================================================
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

from model import PIDeepONet
from physics import compute_stresses
from analytical_kirsch import analytical_kirsch_stress

ARM_TO_PLOT = "rar_combined"  # Select arm to evaluate
model_path = os.path.join(results_dir, f"{ARM_TO_PLOT}_best.pt")
if not os.path.exists(model_path):
    model_path = os.path.join(results_dir, f"{ARM_TO_PLOT}_final.pt")

if not os.path.exists(model_path):
    print(f"⚠️ Checkpoint model for {ARM_TO_PLOT} not found in {results_dir}")
else:
    print(f"Loading model checkpoint: {model_path}")
    config_path = os.path.join(project_root, "configs", f"{ARM_TO_PLOT}.yaml")
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
        
    branch_layers = config["model"]["branch_layers"]
    trunk_layers = config["model"]["trunk_layers"]
    num_sensors = branch_layers[0]
    
    model = PIDeepONet(branch_layers=branch_layers, trunk_layers=trunk_layers, num_outputs=2)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    
    L, R = 1.0, 0.2
    grid_res = 150
    x = np.linspace(-L, L, grid_res)
    y = np.linspace(-L, L, grid_res)
    XX, YY = np.meshgrid(x, y)
    coords_flat = np.column_stack([XX.ravel(), YY.ravel()])
    
    mask_outside = (coords_flat[:, 0]**2 + coords_flat[:, 1]**2) >= R**2
    valid_coords = coords_flat[mask_outside]
    
    T_val = 1.0
    load_tensor = torch.ones(1, num_sensors, dtype=torch.float32) * T_val
    coords_tensor = torch.tensor(valid_coords, dtype=torch.float32, requires_grad=True)
    
    pred = model(load_tensor, coords_tensor)
    u_pred = pred[0, :, 0:1]
    v_pred = pred[0, :, 1:2]
    stresses = compute_stresses(coords_tensor, u_pred, v_pred)
    
    sxx_pred_valid = stresses["sigma_xx"].detach().numpy().squeeze()
    sxx_true_valid, _, _ = analytical_kirsch_stress(valid_coords[:, 0], valid_coords[:, 1], R=R, T=T_val)
    
    def build_grid(valid_vals):
        grid = np.full(XX.shape, np.nan)
        grid[mask_outside.reshape(XX.shape)] = valid_vals
        return grid
        
    sxx_pred_grid = build_grid(sxx_pred_valid)
    sxx_true_grid = build_grid(sxx_true_valid)
    err_sxx_grid = build_grid(np.abs(sxx_pred_valid - sxx_true_valid))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    im0 = axes[0].contourf(XX, YY, sxx_pred_grid, levels=50, cmap="viridis")
    axes[0].set_title(f"PI-DeepONet ({ARM_TO_PLOT}) $\sigma_{{xx}}$")
    axes[0].set_aspect("equal")
    fig.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].contourf(XX, YY, sxx_true_grid, levels=50, cmap="viridis")
    axes[1].set_title("Exact Kirsch Solution $\sigma_{xx}$")
    axes[1].set_aspect("equal")
    fig.colorbar(im1, ax=axes[1])
    
    im2 = axes[2].contourf(XX, YY, err_sxx_grid, levels=50, cmap="magma")
    axes[2].set_title("Absolute Error $|\sigma_{xx}^{pred} - \sigma_{xx}^{true}|$")
    axes[2].set_aspect("equal")
    fig.colorbar(im2, ax=axes[2])
    
    for ax in axes:
        circle = plt.Circle((0, 0), R, color="white", zorder=10)
        ax.add_patch(circle)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        
    plt.suptitle("2D Stress Field Verification — Model vs Analytical Kirsch", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
